# Chapter 2

In [ ]:
import torch

print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")
    device = torch.device("mps")

else:
    print("Only CPU")

PyTorch version 2.10.0
Apple Silicon GPU


In [3]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

In [4]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_path)

In [5]:
prompt = "Tell me a story"
input_token_ids_list = tokenizer.encode(prompt)

In [6]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Tell me a story


In [8]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")


40451 --> Tell
752 -->  me
264 -->  a
3364 -->  story


In [9]:
gibberish = "Me kqwazak moe dubler dar a sturgi muz pho lafa wek gahmen"
input_token_ids_list = tokenizer.encode(gibberish)

for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

7823 --> Me
595 -->  k
69291 --> qw
1370 --> az
585 --> ak
4544 -->  mo
68 --> e
22113 -->  dub
1536 --> ler
15269 -->  dar
264 -->  a
357 -->  st
5556 --> urg
72 --> i
296 -->  m
5197 --> uz
1319 -->  ph
78 --> o
1187 -->  la
3632 --> fa
582 -->  we
74 --> k
342 -->  g
1466 --> ah
5676 --> men


In [ ]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")
        device = torch.device("cuda")
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"  # type: ignore
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True
    elif torch.backends.mps.is_available():
        print("Apple Silicon GPU")
        device = torch.device("mps")
    else:
        print("Only CPU")
        device = torch.device("cpu")
    return device

In [11]:
device = get_device()

Apple Silicon GPU


In [13]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [ ]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)  # instantiate from random weights
model.load_state_dict(torch.load(model_path))  # load pretrained weights
model.to(device)  # transfer model in device memory

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [16]:
prompt = "Explain what an integral is."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)  # convert list to tensor
input_tensor_fmt = input_tensor.unsqueeze(0)  # add a dim
input_tensor_fmt = input_tensor_fmt.to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)  # generate output
output_tensor_fmt = output_tensor.squeeze(0)  # remove the dim
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 7
Formatted Output tensor shape: torch.Size([7, 151936])


### A note on squeezing and unsqueezing Tensors
The pytorch methods `.squeeeze()` and `.unsqueeze()` are useful for resizing tensors to be compatible with other methods or objects that expect a specific dimension, such as the model above which expects two dimensions for rows and columns. The methods always add a dimension of size 1 from the position specified, so in our usage that is 0 when squeezed it adds the dimension and unsqueezing removes this dimension.

In [21]:
print(output_tensor_fmt)

tensor([[ 7.4062, 11.4375,  9.2500,  ...,  3.7500,  3.7500,  3.7500],
        [ 9.4375, 10.6875,  7.2812,  ...,  3.2344,  3.2344,  3.2344],
        [ 8.4375, 11.0000,  5.0312,  ...,  2.5469,  2.5469,  2.5469],
        ...,
        [ 8.6250, 12.7500, 10.6875,  ..., -0.6016, -0.6016, -0.6016],
        [15.5000, 12.9375, 10.6250,  ...,  0.3340,  0.3340,  0.3340],
        [ 9.5000,  5.5000,  7.8438,  ..., -1.4453, -1.4453, -1.4453]],
       device='mps:0', dtype=torch.bfloat16)


In order to get the output we take the last row from this formatted output tensor and find the max value, since the row represents a distribution of potential outputs (a score for every token in the vocabulary) - the max score is the most likely output.

In [17]:
last_row = output_tensor_fmt[-1]
print(last_row)

tensor([ 9.5000,  5.5000,  7.8438,  ..., -1.4453, -1.4453, -1.4453],
       device='mps:0', dtype=torch.bfloat16)


In [33]:
max_token_id = torch.argmax(last_row, dim=-1, keepdim=True)
print(max_token_id)

tensor([2585], device='mps:0')


In [34]:
output = tokenizer.decode([max_token_id])
print(output)

 How


#### Max vs. Argmax
Here we used `torch.argmax` over `torch.max` because the former returns the index of the larges value in a tensor while the latter returns its value.

In [35]:
def text_generation(prompt: str):
    while "<|im_end|>" not in prompt:
        input_token_ids_list = tokenizer.encode(prompt)
        print(f"Number of input tokens: {len(input_token_ids_list)}")

        input_tensor = torch.tensor(input_token_ids_list)  # convert list to tensor
        input_tensor_fmt = input_tensor.unsqueeze(0)  # add a dim
        input_tensor_fmt = input_tensor_fmt.to(device)

        with torch.inference_mode():
            output_tensor = model(input_tensor_fmt)  # generate output
        output_tensor_fmt = output_tensor.squeeze(0)  # remove the dim
        print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

        last_row = output_tensor_fmt[-1]
        print(last_row)

        max_token_id = torch.argmax(last_row, dim=-1, keepdim=True)
        print(max_token_id)

        output = tokenizer.decode([max_token_id])
        print(output)

        prompt += output

        return text_generation(prompt)
    return prompt

In [ ]:
prompt = "Explain integrals."
final_output = text_generation